# Utility Supply-Chain Risk Analysis

A curated version of the analysis notebook prepared for a **team MMA 608 course assignment based on an Ivey case concerning BC Hydro**. The repository author was responsible for the analysis component. This is a historical academic case study, not a commissioned engagement or a statement about the utility's current operations.

The notebook preserves the original inventory screens, category mappings, Composite Vulnerability Score (CVS) weights, fixed 35% tariff scenario, Canadian import-share analysis, and restriction-flag supplier screen. It omits the original cross-country MFN tariff and Doing Business comparisons, whose trade-direction interpretations were unsuitable for assessing sourcing into Canada.

The case workbook is **not distributed**. Outputs and notebook metadata have been cleared. The portfolio figures are separate historical case illustrations, not newly executed results or current forecasts.

In [ ]:
from pathlib import Path
import pandas as pd

FILE = Path("../data/private/Supplement_data.xlsx")
if not FILE.is_file():
    raise FileNotFoundError(
        "The Ivey course-case workbook is not distributed with this repository. "
        "Place an authorized copy at ../data/private/Supplement_data.xlsx and "
        "run this notebook with notebooks/ as its working directory."
    )

vendors_all = pd.read_excel(FILE, sheet_name="All Vendor Data")
risk_tol = pd.read_excel(FILE, sheet_name="Risk Tolerance by Category")
inventory = pd.read_excel(FILE, sheet_name="Vendor and Inventory Data")
imports = pd.read_excel(FILE, sheet_name="Imports from Trading Partners")

inventory.columns = inventory.columns.str.strip()
imports.columns = imports.columns.str.strip()

print(f"Loaded {len(inventory)} inventory rows and {len(vendors_all)} vendor records.")

## 1. Inventory coverage and category exposure

Coverage is days of supply divided by average lead time; the gap is their difference in days. The original At_Risk coverage flag means DoS < LT, not proof of a present stock-out. Likewise, active stock below the supplied reorder point is a screening result: open orders and inventory position are not modeled. Safety stock and reorder points are supplied inputs, not recalculated service-level guarantees.

Category exposure uses the **share of inventory rows with U.S. origin**. It is not a spend-weighted share or a measure of the complete vendor population. The original tariff scenario multiplies annual category spend by that row share and 35%; the result depends on this allocation proxy. The 35% rate is a fixed case assumption, not a current tariff determination.

In [ ]:
# ============================================================
# 2. INVENTORY RISK METRICS (per item)
# ============================================================
inv = inventory.dropna(subset=["Country of Origin"]).copy()
inv.rename(columns={
    "Average Lead Time (days)": "LT", "Standard Deviation of Lead Time (days)": "StdLT",
    "Safety Stock": "SS", "Active Stock (unassigned)": "Active",
    "Days of Supply (current)": "DoS", "Frequency of Use": "Freq",
    "Reorder Point (stock level)": "ROP", "Category NAME": "Category",
    "Country of Origin": "Country", "RestrictedUSVendor": "US_Restricted"
}, inplace=True)

inv["Coverage"] = inv["DoS"] / inv["LT"]          # <1 = at risk
inv["Gap"] = inv["DoS"] - inv["LT"]               # negative = coverage shortfall versus average lead time
inv["CV_LT"] = inv["StdLT"] / inv["LT"]           # relative lead-time variability
inv["Buffer%"] = inv["SS"] / inv["Active"] * 100   # safety stock as % of active
inv["Below_ROP"] = inv["Active"] < inv["ROP"]      # screening flag; open orders are not modeled
inv["At_Risk"] = inv["DoS"] < inv["LT"]

print(f"\nTotal items: {len(inv)}, US-origin inventory rows: {(inv.Country=='USA').sum()}, "
      f"Coverage flags (all): {inv.At_Risk.sum()}, Coverage flags (US): {inv[inv.Country=='USA'].At_Risk.sum()}")

# ============================================================
# 3. CATEGORY-LEVEL AGGREGATION
# ============================================================
# Map short inventory names to risk tolerance full names
CAT_MAP = {
    'Aux Elec Equip': 'Auxiliary Electrical Equipment',
    'Battery Energy Strge': 'Battery Energy Storage System',
    'Chemicals and Gases': 'Chemicals and Gases',
    'Conductor Supp Struc': 'Conductor Support Structures',
    'Construct Material': 'Construction Materials',
    'Construction Equip': 'Construction Equipment Services',
    'Distribution Transf': 'Distribution Transformers',
    'EV Equip & Services': 'Electric Vehicle Equipment & Services',
    'Electrical Component': 'Electrical Components',
    'Fleet': 'Fleet',
    'Fuel, Oil, Lubricant': 'Fuel, Oil and Lubricants',
    'IT': 'IT',
    'MRO-Gen Indus/Safety': 'MRO',
    'Major Gen Powerhouse': 'Major Generation Powerhouse Equipment',
    'Metering & Network': 'Metering & Network devices',
    'Meters': 'Metering & Network devices',
    'Oth Gen Stn Equip': 'Other Generation Station Equipment',
    'Power Transformer': 'Station Transformers',
    'Pre-Cast Concrete': 'Precast Concrete',
    'Protect/ControlEquip': 'Protection & Control Equipment',
    'Switchgear': 'Switchgear',
    'Telecom Equip': 'Grid Telecommunications',
    'Utility Tools/Equip': 'MRO',
    'Wire And Cable': 'Wire & Cable',
}

risk_dict = dict(zip(risk_tol["CATEGORY"], risk_tol["Risk Tolerance of the Category"]))
spend_dict = dict(zip(risk_tol["CATEGORY"], risk_tol["Annual Spend"]))
portfolio_dict = dict(zip(risk_tol["CATEGORY"], risk_tol["PORTFOLIO"]))

inv["RiskTol"] = inv["Category"].map(CAT_MAP).map(risk_dict)
inv["AnnualSpend"] = inv["Category"].map(CAT_MAP).map(spend_dict)
inv["Portfolio"] = inv["Category"].map(CAT_MAP).map(portfolio_dict)

# Category stats
cat = inv.groupby("Category").agg(
    n=("LT", "size"),
    n_us=("Country", lambda x: (x == "USA").sum()),
    at_risk=("At_Risk", "sum"),
    at_risk_us=("At_Risk", lambda x: x[inv.loc[x.index, "Country"] == "USA"].sum()),
    avg_lt=("LT", "mean"),
    avg_dos=("DoS", "mean"),
    avg_cv=("CV_LT", "mean"),
).assign(us_pct=lambda d: d.n_us / d.n * 100)

# Merge risk/spend info
cat["RiskTol"] = cat.index.map(lambda c: risk_dict.get(CAT_MAP.get(c, ""), "N/A"))
cat["Spend"] = cat.index.map(lambda c: spend_dict.get(CAT_MAP.get(c, ""), 0))
cat["Portfolio"] = cat.index.map(lambda c: portfolio_dict.get(CAT_MAP.get(c, ""), "N/A"))

# US-exposed categories only
us_cat = cat[cat.n_us > 0].copy()
us_cat["tariff_impact"] = us_cat["Spend"] * (us_cat["n_us"] / us_cat["n"]) * 0.35

print("\n=== Category Exposure by US-Origin Inventory Rows ===")
print(us_cat[["n", "n_us", "us_pct", "at_risk_us", "avg_lt", "avg_dos", "RiskTol",
              "Spend", "tariff_impact"]].to_string())
print(f"\nIllustrative annual 35% scenario exposure: ${us_cat.tariff_impact.sum():,.0f}")

## 2. Composite Vulnerability Score

The original judgmental weights are retained: **25% U.S.-origin row share, 25% risk tolerance, 20% coverage, 15% presence of a flagged U.S.-origin item, and 15% average lead time**. The cited literature informed the framing; this notebook does not implement an AHP weight-estimation procedure.

The score is a prioritization screen, not a calibrated failure probability. The original "at-risk count" contribution is binary, and category coverage is a ratio of average days of supply to average lead time. Thresholds, weights, and missing-value behavior are preserved for traceability; closely ranked categories should not be interpreted as having decisively different risks.

In [ ]:
# ============================================================
# 4. COMPOSITE VULNERABILITY SCORE (informed by Singh et al. 2021)
# ============================================================
# Weight rationale (informed by Singh et al. 2021 AHP priority ranking):
#   Tier 1 — Supplier-related dimensions (highest AHP priority):
#     - US-origin inventory-row share:  25%  ("critical component supplier")
#     - Risk tolerance level:    25%  ("supplier location" + org risk appetite)
#   Tier 2 — Inventory buffer dimension:
#     - Coverage ratio (DoS/LT): 20%  ("inventory buffering capability")
#   Tier 3 — Exposure & lead time dimensions:
#     - Presence of a coverage flag:      15%  (binary coverage screen)
#     - Average lead time:       15%  ("long supply chain lead time")

def cvs(row):
    """Composite Vulnerability Score: 0 (low) to 1 (high vulnerability)."""
    rt = {"Low": 3, "Med": 2, "High": 1}.get(row.RiskTol, 0) / 3
    cov = 1 - min(row.avg_dos / max(row.avg_lt, 1), 1.5) / 1.5
    ar = min(row.at_risk_us, 1)  # binary: has a US-origin inventory coverage flag
    lt = min(row.avg_lt, 1000) / 1000
    return row.us_pct / 100 * 0.25 + rt * 0.25 + cov * 0.20 + ar * 0.15 + lt * 0.15

us_cat["CVS"] = us_cat.apply(cvs, axis=1)
us_cat = us_cat.sort_values("CVS", ascending=False)
print("\n=== Composite Vulnerability Score ===")
print(us_cat[["CVS", "us_pct", "RiskTol", "at_risk_us", "avg_lt"]].to_string())

## 3. Portfolio spending context

This summary groups the supplied category budgets by portfolio and risk tolerance. It preserves the original aggregation, including exclusion of rows with missing values. Portfolio totals describe the case inputs; they do not add transaction-level evidence to the tariff scenario.

In [ ]:
# ============================================================
# 9. PORTFOLIO-LEVEL SUMMARY
# ============================================================
port = risk_tol.dropna().groupby("PORTFOLIO").apply(
    lambda g: pd.Series({
        "total_spend": g["Annual Spend"].sum(),
        "low_spend": g.loc[g["Risk Tolerance of the Category"] == "Low", "Annual Spend"].sum(),
        "med_spend": g.loc[g["Risk Tolerance of the Category"] == "Med", "Annual Spend"].sum(),
        "high_spend": g.loc[g["Risk Tolerance of the Category"] == "High", "Annual Spend"].sum(),
    })
)
port["low_pct"] = port.low_spend / port.total_spend * 100
print("\n=== Historical Case Portfolio Spend Summary ===")
print(port.to_string())

## 4. Canadian import dependence

Unlike the omitted cross-country tariff comparison, this calculation explicitly holds **Canada as the reporting importer** and compares U.S.-partner imports with World-partner imports for each year and HS code. It provides national trade context, not the utility's purchasing share. The original grouping and missing-value behavior are retained.

In [ ]:
# ============================================================
# 7. EXTERNAL DATA — Canada imports from US by HS code
# ============================================================
ca_imp = imports[imports.reporter_name == "Canada"].copy()
ca_imp["HS Code"] = ca_imp["HS  Code"] if "HS  Code" in ca_imp.columns else ca_imp["HS Code"]
ca_us = ca_imp[ca_imp.partner_name == "United States of America"].groupby(["year", "HS Code"])["value"].sum()
ca_world = ca_imp[ca_imp.partner_name == "World"].groupby(["year", "HS Code"])["value"].sum()
us_share = (ca_us / ca_world * 100).unstack("HS Code")
print("\n=== Canada Import US Share (%) ===")
print(us_share.round(1))

## 5. Supplier screening by restriction flag

This section counts vendor records by the case's RestrictedUSVendor flag and performance rating. The original internal variable is_US represents **flag equals Y**; it is not an independently verified country-of-origin classification. Its complement means records without that flag, not necessarily non-U.S. suppliers. Counts are records, not verified unique companies.

All observed performance-rating columns are retained, including Poor when present. A rating distribution can support an initial screen, but it does not establish technical qualification, available capacity, or interchangeability of equipment.

In [ ]:
# ============================================================
# 6. VENDOR RECORDS — restriction flag and performance
# ============================================================
va = vendors_all.copy()
va["is_US"] = va["RestrictedUSVendor"] == "Y"
va_cat = va.groupby("Category NAME").agg(
    total=("is_US", "size"), us=("is_US", "sum")
).assign(non_us=lambda d: d.total - d.us, us_pct=lambda d: d.us / d.total * 100)

# Performance of records without the US restriction flag in key categories
KEY_CATS = ['Switchgear', 'Distribution Transf', 'Electrical Component', 'IT',
            'Telecom Equip', 'Wire And Cable', 'Protect/ControlEquip', 'Meters',
            'Oth Gen Stn Equip', 'EV Equip & Services', 'MRO-Gen Indus/Safety']
non_us = va[(~va.is_US) & (va["Category NAME"].isin(KEY_CATS))]
perf = non_us.groupby(["Category NAME", "Vendor performance"]).size().unstack(fill_value=0)
print("\n=== Performance of Vendor Records Without the US Restriction Flag ===")
print(perf)

## Portfolio figures and reading notes

The public figures were prepared separately from saved case aggregates. They do not require the private workbook to view:

1. [Category priority scores](../assets/01_priority_scores.png)
2. [Inventory coverage gaps](../assets/02_coverage_gaps.png)
3. [Illustrative 35% tariff scenario](../assets/03_tariff_scenario.png)
4. [Canadian import dependence](../assets/04_import_dependence.png)
5. [Supplier restriction-flag screen](../assets/05_supplier_screen.png)

Read these as a concise historical decision-support case: inventory screens identify questions for follow-up; the CVS organizes priorities; the tariff scenario illustrates sensitivity to a specified shock; and trade/vendor context informs further due diligence. No predictive validation, cost optimization, supplier qualification, or current-policy verification is claimed.